```
fuss/
├── features.pkl   -- {sample_id: {"audio_path": <path_relative_to_dataset_root>}}
├── targets.pkl    -- {sample_id: {"source_audio_paths": (<rel_path_0>, ...), "num_sources": <int>}}
├── train_keys.pkl -- {idx: {"key": sample_id}, ...}
├── val_keys.pkl   -- {idx: {"key": sample_id}, ...}
├── test_keys.pkl  -- {idx: {"key": sample_id}, ...}
└── manifest.csv   -- flat table for quick inspection
```


In [1]:
import os
import pickle
import subprocess
import tarfile
import zipfile
from pathlib import Path

import pandas as pd
import soundfile as sf
from IPython.display import display
from tqdm.auto import tqdm


/Users/nsborodin/repos/autodition/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Setup


In [2]:
repo_root = Path('.').resolve().parent

def load_dotenv(dotenv_path: Path) -> None:
    if not dotenv_path.exists():
        return

    for raw_line in dotenv_path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        key = key.strip()
        value = value.strip().strip('\"').strip("'")
        os.environ.setdefault(key, value)

load_dotenv(repo_root / '.env')

data_dir = Path(os.environ.get('DATA_ROOT', repo_root / 'data'))
raw_data_dir = Path(os.environ.get('RAW_DATA_DIR', data_dir / 'raw'))
preprocessed_data_dir = Path(os.environ.get('PREPROCESSED_DATA_DIR', data_dir / 'preprocessed'))

raw_dir = raw_data_dir / 'FUSS'
download_dir = raw_dir / '_downloads'
preprocessed_dir = preprocessed_data_dir / 'fuss'

kaggle_dataset = 'freecolabgpu/audio-source-separation'
download_from_kaggle = False
extract_archives = False
force_overwrite = False
default_split_ratios = {'train': 0.8, 'val': 0.1, 'test': 0.1}

split_map = {
    'train': 'train',
    'training': 'train',
    'validation': 'val',
    'valid': 'val',
    'val': 'val',
    'dev': 'val',
    'eval': 'test',
    'test': 'test',
    'testing': 'test',
}

audio_extensions = {'.wav', '.flac', '.ogg', '.mp3'}

print(f'Raw dir:          {raw_dir}')
print(f'Download dir:     {download_dir}')
print(f'Preprocessed dir: {preprocessed_dir}')
print(f'Kaggle dataset:   {kaggle_dataset}')
print('Kaggle credentials loaded:', bool(os.environ.get('KAGGLE_USERNAME')) and bool(os.environ.get('KAGGLE_KEY')))


Raw dir:          /Users/nsborodin/repos/autodition/data/raw/FUSS
Download dir:     /Users/nsborodin/repos/autodition/data/raw/FUSS/_downloads
Preprocessed dir: /Users/nsborodin/repos/autodition/data/preprocessed/fuss
Kaggle dataset:   freecolabgpu/audio-source-separation
Kaggle credentials loaded: True


## Download


In [3]:
raw_dir.mkdir(parents=True, exist_ok=True)
download_dir.mkdir(parents=True, exist_ok=True)

if download_from_kaggle:
    missing = [name for name in ('KAGGLE_USERNAME', 'KAGGLE_KEY') if not os.environ.get(name)]
    if missing:
        raise RuntimeError(
            'Missing Kaggle credentials: ' + ', '.join(missing) + '\n'
            'Add them to the project .env or export them before starting Jupyter.'
        )

    subprocess.run(
        [
            'kaggle',
            'datasets',
            'download',
            '-d',
            kaggle_dataset,
            '-p',
            str(download_dir),
            '--unzip',
        ],
        check=True,
        env=os.environ.copy(),
    )
else:
    print('Skipping Kaggle download. Set download_from_kaggle=True to enable it.')


Skipping Kaggle download. Set download_from_kaggle=True to enable it.


In [4]:
archives = sorted(download_dir.glob('*.zip')) + sorted(download_dir.glob('*.tar.gz')) + sorted(download_dir.glob('*.tgz'))
print(f'Found {len(archives)} archive(s) in {download_dir}')
for archive_path in archives:
    print('-', archive_path.name)

if extract_archives:
    for archive_path in archives:
        print(f'Extracting {archive_path.name} ...')
        if archive_path.suffix == '.zip':
            with zipfile.ZipFile(archive_path) as archive:
                archive.extractall(raw_dir)
        elif archive_path.name.endswith('.tar.gz') or archive_path.suffix == '.tgz':
            with tarfile.open(archive_path) as archive:
                archive.extractall(raw_dir)
        else:
            print(f'Skipping unsupported archive: {archive_path.name}')
else:
    print('Skipping archive extraction. Set extract_archives=True to enable it.')


Found 0 archive(s) in /Users/nsborodin/repos/autodition/data/raw/FUSS/_downloads
Skipping archive extraction. Set extract_archives=True to enable it.


## Inspect


In [5]:
audio_files = [path for path in raw_dir.rglob('*') if path.is_file() and path.suffix.lower() in audio_extensions]
print(f'Audio files found: {len(audio_files)}')
for path in audio_files[:20]:
    print(path.relative_to(raw_dir))


Audio files found: 3527
_downloads/mixed_dataset/mixtures/mix_0921.wav
_downloads/mixed_dataset/mixtures/mix_0935.wav
_downloads/mixed_dataset/mixtures/mix_0909.wav
_downloads/mixed_dataset/mixtures/mix_0048.wav
_downloads/mixed_dataset/mixtures/mix_0060.wav
_downloads/mixed_dataset/mixtures/mix_0706.wav
_downloads/mixed_dataset/mixtures/mix_0712.wav
_downloads/mixed_dataset/mixtures/mix_0074.wav
_downloads/mixed_dataset/mixtures/mix_0289.wav
_downloads/mixed_dataset/mixtures/mix_0538.wav
_downloads/mixed_dataset/mixtures/mix_0504.wav
_downloads/mixed_dataset/mixtures/mix_0262.wav
_downloads/mixed_dataset/mixtures/mix_0276.wav
_downloads/mixed_dataset/mixtures/mix_0510.wav
_downloads/mixed_dataset/mixtures/mix_0458.wav
_downloads/mixed_dataset/mixtures/mix_0470.wav
_downloads/mixed_dataset/mixtures/mix_0316.wav
_downloads/mixed_dataset/mixtures/mix_0302.wav
_downloads/mixed_dataset/mixtures/mix_0464.wav
_downloads/mixed_dataset/mixtures/mix_0855.wav


In [6]:
top_level_dirs = sorted([path for path in raw_dir.iterdir() if path.is_dir()])
print('Top-level directories:')
for path in top_level_dirs:
    print('-', path.name)

mix_candidates = [
    path for path in audio_files
    if path.parent.name.lower() in {'mix', 'mixture', 'mixtures'} or path.stem.lower() in {'mix', 'mixture'}
]
print(f'Possible mixture files: {len(mix_candidates)}')
for path in mix_candidates[:10]:
    print('-', path.relative_to(raw_dir))


Top-level directories:
- _downloads
Possible mixture files: 1000
- _downloads/mixed_dataset/mixtures/mix_0921.wav
- _downloads/mixed_dataset/mixtures/mix_0935.wav
- _downloads/mixed_dataset/mixtures/mix_0909.wav
- _downloads/mixed_dataset/mixtures/mix_0048.wav
- _downloads/mixed_dataset/mixtures/mix_0060.wav
- _downloads/mixed_dataset/mixtures/mix_0706.wav
- _downloads/mixed_dataset/mixtures/mix_0712.wav
- _downloads/mixed_dataset/mixtures/mix_0074.wav
- _downloads/mixed_dataset/mixtures/mix_0289.wav
- _downloads/mixed_dataset/mixtures/mix_0538.wav


## Build


In [7]:
import hashlib

def normalize_split(path: Path, sample_key: str) -> str:
    for part in path.parts:
        lowered = part.lower()
        if lowered in split_map:
            return split_map[lowered]

    digest = hashlib.sha1(sample_key.encode('utf-8')).hexdigest()  # nosec B324
    value = int(digest[:8], 16) / 0xFFFFFFFF

    cumulative = 0.0
    for split_name, split_ratio in default_split_ratios.items():
        cumulative += split_ratio
        if value <= cumulative:
            return split_name
    return 'test'


def build_sample_id(mix_path: Path) -> str:
    if mix_path.parent.name.lower() in {'mix', 'mixture', 'mixtures'}:
        return mix_path.stem

    rel = mix_path.relative_to(raw_dir).with_suffix('')
    return rel.as_posix().replace('/', '__')


def find_sources_for_mix(mix_path: Path) -> list[Path]:
    sources = []

    # Layout: .../mixtures/mix_0001.wav and .../sources/mix_0001/*.wav
    if mix_path.parent.name.lower() in {'mix', 'mixture', 'mixtures'}:
        source_dir = mix_path.parent.parent / 'sources' / mix_path.stem
        if source_dir.exists():
            return [
                path
                for path in sorted(source_dir.iterdir())
                if path.is_file() and path.suffix.lower() in audio_extensions
            ]

    # Layout: sibling source directories with the same filename.
    parent = mix_path.parent
    parent_parent = parent.parent
    for sibling_dir in sorted(parent_parent.iterdir()):
        if not sibling_dir.is_dir():
            continue
        name = sibling_dir.name.lower()
        if name in {'mix', 'mixture', 'mixtures'}:
            continue
        if name.startswith('s') and name[1:].isdigit() or name.startswith('source'):
            candidate = sibling_dir / mix_path.name
            if candidate.exists():
                sources.append(candidate)

    if sources:
        return sources

    # Layout: sample-local directory with one mix file and several source files.
    sample_dir = parent_parent if parent.name.lower() in {'mix', 'mixture', 'mixtures'} else parent
    for candidate in sorted(sample_dir.rglob('*')):
        if not candidate.is_file() or candidate.suffix.lower() not in audio_extensions:
            continue
        if candidate == mix_path:
            continue
        if candidate.stem.lower() in {'mix', 'mixture'}:
            continue
        sources.append(candidate)

    return sources


rows = []
for mix_path in tqdm(mix_candidates, desc='Building manifest'):
    source_paths = find_sources_for_mix(mix_path)
    if not source_paths:
        continue

    sample_id = build_sample_id(mix_path)
    split = normalize_split(mix_path.relative_to(raw_dir), sample_id)

    rows.append(
        {
            'sample_id': sample_id,
            'split': split,
            'mixture_rel_path': mix_path.relative_to(raw_dir).as_posix(),
            'source_rel_paths': tuple(path.relative_to(raw_dir).as_posix() for path in source_paths),
            'num_sources': len(source_paths),
        }
    )

manifest = pd.DataFrame(rows).sort_values(['split', 'sample_id']).reset_index(drop=True)
print(f'Samples in manifest: {len(manifest)}')
manifest.head()


Building manifest:   0%|          | 0/1000 [00:00<?, ?it/s]

Building manifest: 100%|██████████| 1000/1000 [00:00<00:00, 15011.88it/s]

Samples in manifest: 1000


,sample_id,split,mixture_rel_path,source_rel_paths,num_sources
0,mix_0008,test,_downloads/mixed_dataset/mixtures/mix_0008.wav,(_downloads/mixed_dataset/sources/mix_0008/mus...,3
1,mix_0018,test,_downloads/mixed_dataset/mixtures/mix_0018.wav,(_downloads/mixed_dataset/sources/mix_0018/spe...,2
2,mix_0019,test,_downloads/mixed_dataset/mixtures/mix_0019.wav,(_downloads/mixed_dataset/sources/mix_0019/mus...,4
3,mix_0039,test,_downloads/mixed_dataset/mixtures/mix_0039.wav,(_downloads/mixed_dataset/sources/mix_0039/spe...,3
4,mix_0055,test,_downloads/mixed_dataset/mixtures/mix_0055.wav,(_downloads/mixed_dataset/sources/mix_0055/mus...,4


In [8]:
metadata_rows = []
for row in tqdm(manifest.itertuples(index=False), total=len(manifest), desc='Reading metadata'):
    mixture_info = sf.info(str(raw_dir / row.mixture_rel_path))
    source_infos = [sf.info(str(raw_dir / rel_path)) for rel_path in row.source_rel_paths]

    metadata_rows.append(
        {
            **row._asdict(),
            'mixture_sr': int(mixture_info.samplerate),
            'mixture_channels': int(mixture_info.channels),
            'mixture_duration': float(mixture_info.frames / mixture_info.samplerate),
            'source_srs': tuple(int(info.samplerate) for info in source_infos),
            'source_channels': tuple(int(info.channels) for info in source_infos),
            'source_durations': tuple(float(info.frames / info.samplerate) for info in source_infos),
        }
    )

manifest = pd.DataFrame(metadata_rows)
manifest.head()


Reading metadata: 100%|██████████| 1000/1000 [00:00<00:00, 1595.06it/s]


,sample_id,split,mixture_rel_path,source_rel_paths,num_sources,mixture_sr,mixture_channels,mixture_duration,source_srs,source_channels,source_durations
0,mix_0008,test,_downloads/mixed_dataset/mixtures/mix_0008.wav,(_downloads/mixed_dataset/sources/mix_0008/mus...,3,22050,1,10.0,"(22050, 22050, 22050)","(1, 1, 1)","(10.0, 10.0, 10.0)"
1,mix_0018,test,_downloads/mixed_dataset/mixtures/mix_0018.wav,(_downloads/mixed_dataset/sources/mix_0018/spe...,2,22050,1,10.0,"(22050, 22050)","(1, 1)","(10.0, 10.0)"
2,mix_0019,test,_downloads/mixed_dataset/mixtures/mix_0019.wav,(_downloads/mixed_dataset/sources/mix_0019/mus...,4,22050,1,10.0,"(22050, 22050, 22050, 22050)","(1, 1, 1, 1)","(10.0, 10.0, 10.0, 10.0)"
3,mix_0039,test,_downloads/mixed_dataset/mixtures/mix_0039.wav,(_downloads/mixed_dataset/sources/mix_0039/spe...,3,22050,1,10.0,"(22050, 22050, 22050)","(1, 1, 1)","(10.0, 10.0, 10.0)"
4,mix_0055,test,_downloads/mixed_dataset/mixtures/mix_0055.wav,(_downloads/mixed_dataset/sources/mix_0055/mus...,4,22050,1,10.0,"(22050, 22050, 22050, 22050)","(1, 1, 1, 1)","(10.0, 10.0, 10.0, 10.0)"


## EDA


In [9]:
print('Samples by split:')
display(manifest['split'].value_counts().rename_axis('split').to_frame('num_samples'))

print('Samples by number of sources:')
display(manifest['num_sources'].value_counts().sort_index().rename_axis('num_sources').to_frame('num_samples'))

print('Mixture sample rates:')
display(manifest['mixture_sr'].value_counts().sort_index().rename_axis('sample_rate').to_frame('num_samples'))

print('Mixture duration summary:')
display(manifest['mixture_duration'].describe().to_frame('value'))


Samples by split:


,num_samples
split,
train,806
test,99
val,95


Samples by number of sources:


,num_samples
num_sources,
1,240
2,265
3,223
4,272


Mixture sample rates:


,num_samples
sample_rate,
22050,1000


Mixture duration summary:


,value
count,1000.0
mean,10.0
std,0.0
min,10.0
25%,10.0
50%,10.0
75%,10.0
max,10.0


## Save


In [10]:
features = {}
targets = {}

for row in manifest.itertuples(index=False):
    features[row.sample_id] = {
        'audio_path': row.mixture_rel_path,
    }
    targets[row.sample_id] = {
        'source_audio_paths': tuple(row.source_rel_paths),
        'num_sources': int(row.num_sources),
    }

train_ids = manifest.loc[manifest['split'] == 'train', 'sample_id'].tolist()
val_ids = manifest.loc[manifest['split'] == 'val', 'sample_id'].tolist()
test_ids = manifest.loc[manifest['split'] == 'test', 'sample_id'].tolist()

train_keys = {idx: {'key': sample_id} for idx, sample_id in enumerate(train_ids)}
val_keys = {idx: {'key': sample_id} for idx, sample_id in enumerate(val_ids)}
test_keys = {idx: {'key': sample_id} for idx, sample_id in enumerate(test_ids)}

preprocessed_dir.mkdir(parents=True, exist_ok=True)

artifacts = {
    preprocessed_dir / 'features.pkl': features,
    preprocessed_dir / 'targets.pkl': targets,
    preprocessed_dir / 'train_keys.pkl': train_keys,
    preprocessed_dir / 'val_keys.pkl': val_keys,
    preprocessed_dir / 'test_keys.pkl': test_keys,
}

for artifact_path, payload in artifacts.items():
    if artifact_path.exists() and not force_overwrite:
        raise FileExistsError(f'{artifact_path} already exists. Set force_overwrite=True to replace it.')
    with open(artifact_path, 'wb') as stream:
        pickle.dump(payload, stream)

manifest.to_csv(preprocessed_dir / 'manifest.csv', index=False)

print(f'Saved features:  {len(features)} entries -> {preprocessed_dir / "features.pkl"}')
print(f'Saved targets:   {len(targets)} entries -> {preprocessed_dir / "targets.pkl"}')
print(f'Saved train_keys:{len(train_keys):>4} entries -> {preprocessed_dir / "train_keys.pkl"}')
print(f'Saved val_keys:  {len(val_keys):>4} entries -> {preprocessed_dir / "val_keys.pkl"}')
print(f'Saved test_keys: {len(test_keys):>4} entries -> {preprocessed_dir / "test_keys.pkl"}')
print(f'Saved manifest:  {len(manifest)} rows -> {preprocessed_dir / "manifest.csv"}')


Saved features:  1000 entries -> /Users/nsborodin/repos/autodition/data/preprocessed/fuss/features.pkl
Saved targets:   1000 entries -> /Users/nsborodin/repos/autodition/data/preprocessed/fuss/targets.pkl
Saved train_keys: 806 entries -> /Users/nsborodin/repos/autodition/data/preprocessed/fuss/train_keys.pkl
Saved val_keys:    95 entries -> /Users/nsborodin/repos/autodition/data/preprocessed/fuss/val_keys.pkl
Saved test_keys:   99 entries -> /Users/nsborodin/repos/autodition/data/preprocessed/fuss/test_keys.pkl
Saved manifest:  1000 rows -> /Users/nsborodin/repos/autodition/data/preprocessed/fuss/manifest.csv
